# F2 · Obtención, exploración, limpieza, transformación y validación

**Proyecto:** Análisis reproducible del Inventario Público de Glaciares de Chile 2022
**Grupo 3 · MCDI500 — Programación para la Ciencia de Datos**
**Integrantes:** Aníbal Reyes · Francisco Correa Soto · Gustavo Aloy · Roberto Gaubert

**Pregunta:** ¿Cómo se distribuyen los glaciares del Inventario Público de Glaciares 2022 según región, clasificación, superficie y volumen estimado?

**Propósito de este notebook.** Convertir el inventario original en un conjunto limpio, transformado y validado para la Fase 3, siguiendo los pasos 6 a 11 del pipeline del curso:

| § | Etapa | Producto |
|---|---|---|
| 0–1 | Entorno, bitácora y obtención | Fuente verificada con su huella SHA-256 |
| 2 | Exploración y medición | Cifras que justifican cada decisión |
| 3 | Limpieza e imputación | Columnas sin información descartadas, texto normalizado, tipos corregidos, faltantes encubiertos tratados |
| 4 | Transformación por tipo de variable | Derivadas, ordinales, cíclica y one-hot |
| 5 | Escalamiento | Comparación de tres escaladores y elección justificada |
| 6 | Validación y pruebas | Asserts del dataset y pruebas de casos normales, límite y excepciones |
| 7 | Persistencia y bitácora | `data/processed/` y `docs/bitacora_f2.md` |

**Regla de trabajo:** medir primero, decidir después y justificar con la medición. El archivo de `data/raw/` nunca se modifica, y ningún glaciar se elimina sin evidencia.

## 0. Entorno, rutas y bitácora

Se cargan las librerías del pipeline y se localiza la raíz del repositorio con **rutas relativas**, con la misma función que en F1. El notebook funciona desde `F2/` o desde la raíz. También se verifica que el kernel sea el del entorno virtual del proyecto.

La clase `Bitacora` anota cada decisión **en el momento en que se toma, con sus cifras**. Al final se exporta a `docs/bitacora_f2.md`, de modo que el informe usa exactamente los mismos números que el notebook.

In [1]:
import sys
import re
import hashlib
import subprocess
import unicodedata
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

NOMBRE_ARCHIVO = "Inventario_Publico_de_Glaciares_2022.csv"
HUELLA_REFERENCIA = "5691de5d7d7ffe2c8586a7066e227ac5cc91fa1ec69f54096e937a054c283f1d"

def localizar_raiz(inicio=None):
    """Sube desde `inicio` (o el directorio actual) hasta encontrar data/raw/<NOMBRE_ARCHIVO>."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for carpeta in (inicio, *inicio.parents):
        if (carpeta / "data" / "raw" / NOMBRE_ARCHIVO).is_file():
            return carpeta
    raise FileNotFoundError(f"No se encontró data/raw/{NOMBRE_ARCHIVO} desde {inicio}")

RAIZ = localizar_raiz()
ENTRADA = RAIZ / "data" / "raw" / NOMBRE_ARCHIVO
SALIDA = RAIZ / "data" / "processed"
DOCS = RAIZ / "docs"

declaradas = dict(linea.split("==", 1) for linea in (RAIZ / "requirements.txt").read_text(encoding="utf-8").split()
                  if "==" in linea)
print("Python:", sys.version.split()[0], "| entorno virtual:", "sí" if sys.prefix != sys.base_prefix else "NO")
for paquete in ("numpy", "pandas", "scikit-learn"):
    instalada = metadata.version(paquete)
    aviso = "" if instalada == declaradas.get(paquete) else f"  ← requirements.txt declara {declaradas.get(paquete)}"
    print(f"{paquete}: {instalada}{aviso}")
print("Fuente:", ENTRADA.relative_to(RAIZ))

Python: 3.14.6 | entorno virtual: sí
numpy: 2.5.4
pandas: 3.1.1
scikit-learn: 1.9.1
Fuente: data/raw/Inventario_Publico_de_Glaciares_2022.csv


In [2]:
class Bitacora:
    """Registro ordenado de decisiones del pipeline, cada una con su evidencia cuantitativa."""

    def __init__(self, fase):
        self.fase = fase
        self._registros = []

    def registrar(self, etapa, decision, evidencia):
        """Agrega una decisión; si ya existe (celda re-ejecutada), actualiza su evidencia sin duplicarla."""
        if not str(evidencia).strip():
            raise ValueError("Toda decisión debe registrarse con su evidencia")
        for registro in self._registros:
            if (registro["etapa"], registro["decision"]) == (etapa, decision):
                registro["evidencia"] = evidencia
                break
        else:
            self._registros.append({"n": len(self._registros) + 1, "etapa": etapa,
                                    "decision": decision, "evidencia": evidencia})
        print(f"[{self.fase} · {etapa}] {decision} — {evidencia}")

    def tabla(self):
        return pd.DataFrame(self._registros)

    def exportar_markdown(self, ruta):
        filas = ["| # | Etapa | Decisión | Evidencia |", "|---|---|---|---|"]
        filas += [f"| {r['n']} | {r['etapa']} | {r['decision']} | {r['evidencia']} |" for r in self._registros]
        Path(ruta).write_text(f"# Bitácora de decisiones · {self.fase}\n\n" + "\n".join(filas) + "\n", encoding="utf-8")

    def __len__(self):
        return len(self._registros)

bitacora = Bitacora("F2")

## 1. Obtención y preservación de la fuente

Se calcula la huella SHA-256 del CSV y se compara con la registrada en el README y en F1. Así se garantiza que el pipeline procesa la misma versión del inventario en cualquier equipo. La lectura usa `utf-8-sig` para ignorar una posible marca BOM al inicio del archivo. `original` no se modifica nunca: cada etapa posterior produce **su propio DataFrame** a partir de la anterior (`etapa_columnas` → `etapa_texto` → `etapa_tipos` → `etapa_volumen` → `etapa_marcada` → `etapa_imputada` → `transformado`). Así el flujo de datos es verificable y re-ejecutar cualquier celda suelta produce el mismo resultado.

In [3]:
def huella_sha256(ruta, bloque=1024 * 1024):
    """Huella SHA-256 del archivo, leído por bloques (no lo modifica)."""
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for trozo in iter(lambda: archivo.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()

def cargar_inventario(ruta):
    """Lee el CSV original sin alterar valores."""
    return pd.read_csv(ruta, encoding="utf-8-sig", low_memory=False)

huella_inicial = huella_sha256(ENTRADA)
assert huella_inicial == HUELLA_REFERENCIA, "El CSV no coincide con la versión registrada"
original = cargar_inventario(ENTRADA)

bitacora.registrar("obtención", "Se carga el CSV oficial desde data/raw sin modificarlo",
                   f"{original.shape[0]:,} filas × {original.shape[1]} columnas; SHA-256 {huella_inicial[:12]}… = referencia")

[F2 · obtención] Se carga el CSV oficial desde data/raw sin modificarlo — 26,180 filas × 45 columnas; SHA-256 5691de5d7d7f… = referencia


## 2. Exploración y medición

Antes de tocar los datos se miden los cuatro aspectos que pide el curso: **dimensiones y tipos, valores faltantes, categorías observadas y valores atípicos**. A ellos se suman dos revisiones específicas de este inventario: los **faltantes encubiertos** (ceros que no aparecen como nulos) y la **coherencia física** entre variables. Cada hallazgo queda en la bitácora con su cifra.

### 2.1 Perfil de tipos, faltantes y variables sin información

In [4]:
def perfil_columnas(df):
    """Tipo, no nulos, faltantes y valores únicos de cada columna."""
    return pd.DataFrame({
        "variable": df.columns,
        "tipo": [str(df[c].dtype) for c in df.columns],
        "no_nulos": [int(df[c].notna().sum()) for c in df.columns],
        "pct_faltante": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "valores_unicos": [int(df[c].nunique(dropna=True)) for c in df.columns],
    })

def columnas_sin_informacion(df, umbral_faltantes=0.60):
    """Columnas con más faltantes que el umbral o con un único valor observado."""
    filas = []
    for columna in df.columns:
        pct = df[columna].isna().mean()
        unicos = df[columna].nunique(dropna=True)
        if pct > umbral_faltantes or unicos <= 1:
            filas.append({
                "variable": columna,
                "pct_faltante": round(pct * 100, 2),
                "valores_unicos": unicos,
                "motivo": f"faltantes > {umbral_faltantes:.0%}" if pct > umbral_faltantes else "valor único",
                "valor_observado": df[columna].dropna().iloc[0] if unicos == 1 else None,
            })
    return pd.DataFrame(filas)

perfil_inicial = perfil_columnas(original)
display(perfil_inicial)
sin_info = columnas_sin_informacion(original)
display(sin_info)
print("Filas duplicadas exactas:", int(original.duplicated().sum()))
print("COD_GLA únicos:", original["COD_GLA"].is_unique, "| OBJECTID repetidos:", int(original["OBJECTID"].duplicated().sum()))

for _, fila in sin_info.iterrows():
    bitacora.registrar("exploración", f"{fila['variable']} no aporta información ({fila['motivo']})",
                       f"{fila['pct_faltante']} % faltante; {fila['valores_unicos']} valor(es) único(s)"
                       + (f"; valor = {fila['valor_observado']}" if fila["valor_observado"] is not None else ""))
bitacora.registrar("exploración", "COD_GLA es el identificador; OBJECTID no identifica filas",
                   f"COD_GLA único en {original['COD_GLA'].nunique():,} filas; OBJECTID repetido en "
                   f"{int(original['OBJECTID'].duplicated().sum())} filas; 0 filas duplicadas")

,variable,tipo,no_nulos,pct_faltante,valores_unicos
0,OBJECTID,int64,26180,0.00,26078
1,Id,int64,26180,0.00,1
2,COD_GLA,str,26180,0.00,26180
3,NOMBRE,str,26180,0.00,431
4,CLASIFICA,str,26180,0.00,5
5,AREA_KM2,float64,26180,0.00,2323
6,REGION,str,26180,0.00,16
7,PROVINCIA,str,26180,0.00,40
8,COMUNA,str,26180,0.00,84
9,DATUM,str,26180,0.00,1


,variable,pct_faltante,valores_unicos,motivo,valor_observado
0,Id,0.00,1,valor único,0
1,DATUM,0.00,1,valor único,WGS 84
2,HUSO,0.00,1,valor único,19 SUR
3,CH_CASQ,97.24,6,faltantes > 60%,None
4,CUBIERTO,100.00,0,faltantes > 60%,None
5,F_ESP_MED,0.00,1,valor único,CHEN & OHMURA 1990
6,OBSERVACIO,100.00,0,faltantes > 60%,None


Filas duplicadas exactas: 0
COD_GLA únicos: True | OBJECTID repetidos: 102
[F2 · exploración] Id no aporta información (valor único) — 0.0 % faltante; 1 valor(es) único(s); valor = 0
[F2 · exploración] DATUM no aporta información (valor único) — 0.0 % faltante; 1 valor(es) único(s); valor = WGS 84
[F2 · exploración] HUSO no aporta información (valor único) — 0.0 % faltante; 1 valor(es) único(s); valor = 19 SUR
[F2 · exploración] CH_CASQ no aporta información (faltantes > 60%) — 97.24 % faltante; 6 valor(es) único(s)
[F2 · exploración] CUBIERTO no aporta información (faltantes > 60%) — 100.0 % faltante; 0 valor(es) único(s)
[F2 · exploración] F_ESP_MED no aporta información (valor único) — 0.0 % faltante; 1 valor(es) único(s); valor = CHEN & OHMURA 1990
[F2 · exploración] OBSERVACIO no aporta información (faltantes > 60%) — 100.0 % faltante; 0 valor(es) único(s)
[F2 · exploración] COD_GLA es el identificador; OBJECTID no identifica filas — COD_GLA único en 26,180 filas; OBJECTID repetid

### 2.2 Variables numéricas: rangos, asimetría, ceros y atípicos

El rango intercuartílico (IQR) cuantifica los valores extremos. La asimetría compara media y mediana. Los **ceros** se cuentan aparte porque, en magnitudes físicas, pueden ser faltantes encubiertos.

In [5]:
VARIABLES_FISICAS = ["AREA_KM2", "VOL_km3", "EQ_AGUAKM3", "ESP_MED", "HMIN", "HMEDIA", "HMAX", "PENDIENTE"]

def resumen_numerico(df, columnas):
    """Estadísticos, asimetría, ceros y conteo fuera del rango IQR (1,5 × RIC)."""
    filas = []
    for columna in columnas:
        x = df[columna].dropna()
        q1, q3 = x.quantile([0.25, 0.75])
        ric = q3 - q1
        fuera = int(((x < q1 - 1.5 * ric) | (x > q3 + 1.5 * ric)).sum())
        filas.append({"variable": columna, "media": x.mean(), "mediana": x.median(), "min": x.min(),
                      "max": x.max(), "asimetria": x.skew(), "ceros": int((x == 0).sum()),
                      "fuera_iqr": fuera, "pct_fuera_iqr": round(100 * fuera / len(x), 2)})
    return pd.DataFrame(filas).round(4)

resumen_inicial = resumen_numerico(original, VARIABLES_FISICAS)
display(resumen_inicial)

r = resumen_inicial.set_index("variable")
bitacora.registrar("exploración", "Área y volumen son muy asimétricos, con atípicos legítimos",
                   f"AREA_KM2: asimetría {r.loc['AREA_KM2','asimetria']:.1f}, media {r.loc['AREA_KM2','media']:.3f} vs. "
                   f"mediana {r.loc['AREA_KM2','mediana']:.3f} km², {r.loc['AREA_KM2','pct_fuera_iqr']} % fuera del IQR; "
                   f"VOL_km3: asimetría {r.loc['VOL_km3','asimetria']:.1f}")

,variable,media,mediana,min,max,asimetria,ceros,fuera_iqr,pct_fuera_iqr
0,AREA_KM2,0.8026,0.0360,0.001,772.5110,43.0911,0,3761,14.37
1,VOL_km3,0.1156,0.0003,0.000,236.4457,54.0470,362,4380,16.73
2,EQ_AGUAKM3,0.0983,0.0002,0.000,200.9789,54.0471,668,4377,16.72
3,ESP_MED,12.3631,8.7365,2.127,306.0740,7.0874,0,1758,6.72
4,HMIN,1815.4097,1421.0000,0.000,6726.0000,1.4045,69,5195,19.84
5,HMEDIA,1935.0492,1524.6130,27.940,6734.6310,1.4490,0,5254,20.07
6,HMAX,2047.5790,1616.0000,111.000,6764.0000,1.4120,0,5124,19.57
7,PENDIENTE,25.4616,25.4560,1.119,74.3000,0.1705,0,357,1.36


[F2 · exploración] Área y volumen son muy asimétricos, con atípicos legítimos — AREA_KM2: asimetría 43.1, media 0.803 vs. mediana 0.036 km², 14.37 % fuera del IQR; VOL_km3: asimetría 54.0


### 2.3 Faltantes encubiertos: ceros en volumen y altitud

**Volumen.** El inventario estima el volumen a partir del área y del espesor medio (Chen & Ohmura, 1990). Si esa relación se cumple en los registros con volumen, los `VOL_km3 = 0` pueden reconstruirse con la misma fórmula en vez de tratarse como glaciares sin hielo. La tabla siguiente mide la razón `VOL / (AREA · ESP_MED / 1000)` y `EQ / VOL` por tipo de glaciar, solo en glaciares con volumen ≥ 0,001 km³, para evitar el efecto del redondeo a 5 decimales.

**Altitud mínima.** Un `HMIN = 0` (nivel del mar) solo es físicamente plausible en glaciares que terminan en el mar (frente en **fiordo**). Se cruza con el tipo de frente.

In [6]:
def relacion_volumen(df, minimo=1e-3):
    """Mediana de VOL/(AREA·ESP/1000) y de EQ/VOL por clasificación."""
    m = df["VOL_km3"] >= minimo
    sub = df[m].assign(
        razon_volumen=lambda d: d["VOL_km3"] / (d["AREA_KM2"] * d["ESP_MED"] / 1000),
        razon_agua=lambda d: d["EQ_AGUAKM3"] / d["VOL_km3"],
    )
    return sub.groupby("CLASIFICA")[["razon_volumen", "razon_agua"]].median().round(4)

tabla_volumen = relacion_volumen(original)
display(tabla_volumen)
vol0 = original["VOL_km3"] == 0
eq0 = original["EQ_AGUAKM3"] == 0
print("VOL_km3 = 0:", int(vol0.sum()), "| clasificación:", original.loc[vol0, "CLASIFICA"].value_counts().to_dict(),
      "| área:", sorted(original.loc[vol0, "AREA_KM2"].unique()))
print("EQ_AGUAKM3 = 0:", int(eq0.sum()), "| de ellos con VOL_km3 > 0:", int((eq0 & ~vol0).sum()),
      "| VOL máximo en esos casos:", original.loc[eq0 & ~vol0, "VOL_km3"].max())

hmin_frente = pd.crosstab(original["HMIN"] == 0, original["FRENTE_TER"]).rename(index={False: "HMIN > 0", True: "HMIN = 0"})
display(hmin_frente)
hmin0_tierra = int(((original["HMIN"] == 0) & (original["FRENTE_TER"] != "FIORDO")).sum())

bitacora.registrar("exploración", "VOL_km3 = 0 es redondeo, no ausencia de hielo",
                   f"{int(vol0.sum())} glaciaretes de área ≤ 0,002 km²; la fórmula reproduce el volumen (razón mediana "
                   f"{tabla_volumen.loc['GLACIARETE','razon_volumen']}; rocosos {tabla_volumen.loc['GLACIAR ROCOSO','razon_volumen']} = 60 % de hielo)")
bitacora.registrar("exploración", "EQ_AGUAKM3 es una proporción fija de VOL_km3 según el tipo",
                   f"EQ/VOL mediana {tabla_volumen.loc['GLACIARETE','razon_agua']} (850 kg/m³) y "
                   f"{tabla_volumen.loc['GLACIAR ROCOSO','razon_agua']} en rocosos; {int(eq0.sum())} ceros por redondeo")
bitacora.registrar("exploración", "HMIN = 0 con frente en tierra o lago es un faltante encubierto",
                   f"{int((original['HMIN'] == 0).sum())} glaciares con HMIN = 0: "
                   f"{int(hmin_frente.loc['HMIN = 0','FIORDO'])} en fiordo (plausible) y {hmin0_tierra} en tierra/lago (implausible)")

,razon_volumen,razon_agua
CLASIFICA,,
GLACIAR DE MONTAÑA,1.0000,0.850
GLACIAR DE VALLE,1.0000,0.850
GLACIAR EFLUENTE,1.0000,0.850
GLACIAR ROCOSO,0.6000,0.917
GLACIARETE,0.9999,0.850


VOL_km3 = 0: 362 | clasificación: {'GLACIARETE': 362} | área: [np.float64(0.001), np.float64(0.002)]
EQ_AGUAKM3 = 0: 668 | de ellos con VOL_km3 > 0: 306 | VOL máximo en esos casos: 1e-05


FRENTE_TER,FIORDO,LAGO,TIERRA
HMIN,,,
HMIN > 0,47,664,25400
HMIN = 0,36,8,25


[F2 · exploración] VOL_km3 = 0 es redondeo, no ausencia de hielo — 362 glaciaretes de área ≤ 0,002 km²; la fórmula reproduce el volumen (razón mediana 0.9999; rocosos 0.6 = 60 % de hielo)
[F2 · exploración] EQ_AGUAKM3 es una proporción fija de VOL_km3 según el tipo — EQ/VOL mediana 0.85 (850 kg/m³) y 0.917 en rocosos; 668 ceros por redondeo
[F2 · exploración] HMIN = 0 con frente en tierra o lago es un faltante encubierto — 69 glaciares con HMIN = 0: 36 en fiordo (plausible) y 33 en tierra/lago (implausible)


### 2.4 Coherencia física, redundancias, categorías y fechas

In [7]:
MOJIBAKE = re.compile("[├┤│ÃÂ]")

def clave_texto(valor):
    """Clave de comparación: sin tildes, en mayúsculas y con espacios simples."""
    sin_tildes = unicodedata.normalize("NFKD", str(valor)).encode("ascii", "ignore").decode()
    return " ".join(sin_tildes.upper().split())

def categorias_sospechosas(df, columnas):
    """Detecta codificación rota, mayúsculas o espacios irregulares y variantes de una misma categoría."""
    filas = []
    for columna in columnas:
        valores = pd.Series(df[columna].dropna().unique())
        for valor in valores:
            if MOJIBAKE.search(valor):
                filas.append({"variable": columna, "valores": valor, "problema": "codificación de caracteres"})
            elif valor != valor.upper():
                filas.append({"variable": columna, "valores": valor, "problema": "mayúsculas inconsistentes"})
            elif valor != " ".join(valor.split()):
                filas.append({"variable": columna, "valores": valor, "problema": "espacios irregulares"})
        for _, grupo in valores.groupby(valores.map(clave_texto)):
            if len(grupo) > 1:
                filas.append({"variable": columna, "valores": " | ".join(grupo), "problema": "misma categoría escrita distinto"})
    return pd.DataFrame(filas, columns=["variable", "valores", "problema"])

def es_correspondencia_uno_a_uno(df, a, b):
    """True si cada valor de `a` corresponde a un único valor de `b` y viceversa."""
    return bool((df.groupby(a)[b].nunique() == 1).all() and (df.groupby(b)[a].nunique() == 1).all())

def clasificar_formato_fecha(serie):
    """Etiqueta el formato de cada fecha de texto."""
    texto = serie.astype("string").str.strip()
    dia_mes = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    mes_dia = pd.to_datetime(texto, format="%m/%d/%Y", errors="coerce")
    solo_anio = texto.str.fullmatch(r"\d{4}").fillna(False)
    return pd.Series(np.select([dia_mes.notna().to_numpy(), mes_dia.notna().to_numpy(), solo_anio.to_numpy(dtype=bool)],
                               ["DD/MM/AAAA", "MM/DD/AAAA", "AAAA"], "no interpretable"), index=serie.index)

incoherentes = original.query("not (HMIN <= HMEDIA <= HMAX)")
print("Registros que violan HMIN ≤ HMEDIA ≤ HMAX:", len(incoherentes))
display(incoherentes[["COD_GLA", "REGION", "HMIN", "HMEDIA", "HMAX"]])

wgi_redundante = es_correspondencia_uno_a_uno(original, "CLASIFICA", "WGI_1")
print("WGI_1 ↔ CLASIFICA uno a uno:", wgi_redundante)

COLUMNAS_TEXTO = [c for c in original.columns
                  if pd.api.types.is_string_dtype(original[c]) and original[c].notna().any()
                  and c not in ("COD_GLA", "LATITUD", "LONGITUD", "FUEN_FECHA")]
sospechosas = categorias_sospechosas(original, COLUMNAS_TEXTO)
display(sospechosas)

formatos_fecha = clasificar_formato_fecha(original["FUEN_FECHA"]).value_counts()
display(formatos_fecha.rename("registros").to_frame())
print("LATITUD de ejemplo (texto con coma decimal):", original["LATITUD"].head(2).tolist())

bitacora.registrar("exploración", "Un glaciar viola HMIN ≤ HMEDIA ≤ HMAX",
                   "; ".join(f"{f.COD_GLA} ({f.REGION}): HMEDIA {f.HMEDIA:,.0f} m < HMIN {f.HMIN:,.0f} m" for f in incoherentes.itertuples()))
bitacora.registrar("exploración", "WGI_1 duplica la información de CLASIFICA",
                   f"correspondencia uno a uno entre las {original['CLASIFICA'].nunique()} clases y los códigos WGI_1: {wgi_redundante}")
bitacora.registrar("exploración", "Revisión de categorías: codificación, mayúsculas, espacios y variantes",
                   "; ".join(f"{c}: '{v}' ({p})" for c, v, p in sospechosas[["variable", "valores", "problema"]].itertuples(index=False))
                   or f"sin problemas en {len(COLUMNAS_TEXTO)} columnas de texto")
bitacora.registrar("exploración", "FUEN_FECHA mezcla formatos; LATITUD/LONGITUD son texto con coma decimal",
                   ", ".join(f"{k}: {v:,}" for k, v in formatos_fecha.items()) + "; ejemplo LATITUD '-18,1744803305003'")

Registros que violan HMIN ≤ HMEDIA ≤ HMAX: 1


,COD_GLA,REGION,HMIN,HMEDIA,HMAX
26167,CL108116007@,ÑUBLE,2849.0,2228.215,2937.0


WGI_1 ↔ CLASIFICA uno a uno: True


,variable,valores,problema
0,NOMBRE,VOLCAN PLANCHON PETEROA,espacios irregulares
1,FUENTE_DIG,LiDAR ANDINA,mayúsculas inconsistentes
2,FUENTE_DIG,FOTOGRAMETR├ìA,codificación de caracteres
3,NOM_CUEN,"COSTERAS ENTRE LAG. BLANCA(INC), SENO OTWAY, ...",espacios irregulares


,registros
DD/MM/AAAA,26167
AAAA,12
MM/DD/AAAA,1


LATITUD de ejemplo (texto con coma decimal): ['-18,1744803305003', '-18,1245109467846']
[F2 · exploración] Un glaciar viola HMIN ≤ HMEDIA ≤ HMAX — CL108116007@ (ÑUBLE): HMEDIA 2,228 m < HMIN 2,849 m
[F2 · exploración] WGI_1 duplica la información de CLASIFICA — correspondencia uno a uno entre las 5 clases y los códigos WGI_1: True
[F2 · exploración] Revisión de categorías: codificación, mayúsculas, espacios y variantes — NOMBRE: 'VOLCAN PLANCHON  PETEROA' (espacios irregulares); FUENTE_DIG: 'LiDAR ANDINA' (mayúsculas inconsistentes); FUENTE_DIG: 'FOTOGRAMETR├ìA' (codificación de caracteres); NOM_CUEN: 'COSTERAS ENTRE LAG.  BLANCA(INC), SENO OTWAY, CANAL JERONIMO Y MAGALLANES' (espacios irregulares)
[F2 · exploración] FUEN_FECHA mezcla formatos; LATITUD/LONGITUD son texto con coma decimal — DD/MM/AAAA: 26,167, AAAA: 12, MM/DD/AAAA: 1; ejemplo LATITUD '-18,1744803305003'


## 3. Limpieza e imputación

### 3.1 Columnas sin información y bandera de campo de hielo

El curso fija un criterio: una variable con **más de 60 % de faltantes no es recuperable y se descarta**, y una columna con un único valor no aporta información.

- **Se descartan** `CUBIERTO` y `OBSERVACIO` (100 % vacías) y `Id`, `DATUM`, `HUSO` y `F_ESP_MED`, que tienen un solo valor. Sus valores (WGS 84, huso 19 Sur y método Chen & Ohmura 1990) quedan registrados en la bitácora como metadatos, así que no se pierde información.
- **Se descarta** `WGI_1`, porque duplica uno a uno a `CLASIFICA` (§2.4).
- **Se descarta** `OBJECTID`, que no identifica filas. El identificador es `COD_GLA`, que se conserva como clave pero se **excluye del análisis**, como pide el curso para los identificadores.
- **`CH_CASQ` (97,24 % faltante)** no es recuperable como categoría. Pero su faltante no es información perdida: significa que el glaciar *no pertenece* a un campo de hielo. Ese contenido se conserva en una **variable binaria** `EN_CAMPO_HIELO` (1 si tiene código), y luego se descarta la columna original.

In [8]:
def eliminar_columnas(df, columnas):
    """Devuelve una copia sin las columnas indicadas; falla si alguna no existe."""
    faltan = set(columnas) - set(df.columns)
    if faltan:
        raise KeyError(f"Columnas inexistentes: {sorted(faltan)}")
    return df.drop(columns=list(columnas))

DESCARTAR = list(sin_info.query("variable != 'CH_CASQ'")["variable"]) + ["WGI_1", "OBJECTID"]
etapa_columnas = eliminar_columnas(original.assign(EN_CAMPO_HIELO=original["CH_CASQ"].notna().astype(int)),
                                   DESCARTAR + ["CH_CASQ"])

print("Columnas descartadas:", DESCARTAR + ["CH_CASQ"])
print("Dimensiones:", original.shape, "→", etapa_columnas.shape)
bitacora.registrar("limpieza", f"Se descartan {len(DESCARTAR) + 1} columnas: {', '.join(DESCARTAR + ['CH_CASQ'])}",
                   "100 % vacías (CUBIERTO, OBSERVACIO), valor único (Id, DATUM = WGS 84, HUSO = 19 SUR, "
                   "F_ESP_MED = CHEN & OHMURA 1990), redundante (WGI_1 ≡ CLASIFICA), sin unicidad (OBJECTID), "
                   f"CH_CASQ > 60 % faltante ({original.shape[1]} → {etapa_columnas.shape[1] - 1} columnas originales)")
bitacora.registrar("limpieza", "CH_CASQ se convierte en la bandera binaria EN_CAMPO_HIELO",
                   f"el faltante significa 'no pertenece'; {int(etapa_columnas['EN_CAMPO_HIELO'].sum())} glaciares pertenecen a un campo de hielo o casquete")

Columnas descartadas: ['Id', 'DATUM', 'HUSO', 'CUBIERTO', 'F_ESP_MED', 'OBSERVACIO', 'WGI_1', 'OBJECTID', 'CH_CASQ']
Dimensiones: (26180, 45) → (26180, 37)
[F2 · limpieza] Se descartan 9 columnas: Id, DATUM, HUSO, CUBIERTO, F_ESP_MED, OBSERVACIO, WGI_1, OBJECTID, CH_CASQ — 100 % vacías (CUBIERTO, OBSERVACIO), valor único (Id, DATUM = WGS 84, HUSO = 19 SUR, F_ESP_MED = CHEN & OHMURA 1990), redundante (WGI_1 ≡ CLASIFICA), sin unicidad (OBJECTID), CH_CASQ > 60 % faltante (45 → 36 columnas originales)
[F2 · limpieza] CH_CASQ se convierte en la bandera binaria EN_CAMPO_HIELO — el faltante significa 'no pertenece'; 723 glaciares pertenecen a un campo de hielo o casquete


### 3.2 Normalización del texto

Una misma categoría escrita de dos maneras se contaría como dos grupos en F3. Se repara la codificación rota y se uniformizan mayúsculas y espacios (hay espacios dobles en `NOM_CUEN` y `NOMBRE`) en todas las columnas de texto. La reparación solo se aplica si el valor contiene caracteres típicos de una codificación mal leída: `FOTOGRAMETR├ìA` es `FOTOGRAMETRÍA` guardado en UTF-8 y leído como CP437.

In [9]:
def reparar_codificacion(texto):
    """Revierte UTF-8 leído como CP437 solo si hay señales de codificación rota."""
    if not MOJIBAKE.search(texto):
        return texto
    try:
        return texto.encode("cp437").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return texto

def normalizar_texto(serie):
    """Repara la codificación, pasa a mayúsculas y colapsa espacios; los nulos se mantienen."""
    return serie.map(lambda v: v if pd.isna(v) else " ".join(reparar_codificacion(v).upper().split()))

def normalizar_columnas(df, columnas):
    """Aplica normalizar_texto a `columnas`; devuelve (copia normalizada, cambios por columna)."""
    df = df.copy()
    cambios = {}
    for columna in columnas:
        nueva = normalizar_texto(df[columna])
        cambios[columna] = int((nueva.fillna("") != df[columna].fillna("")).sum())
        df[columna] = nueva
    return df, {c: n for c, n in cambios.items() if n}

etapa_texto, cambios_texto = normalizar_columnas(etapa_columnas, [c for c in COLUMNAS_TEXTO if c in etapa_columnas.columns])
print("Valores corregidos por columna:", cambios_texto)
print("FUENTE_DIG:", sorted(etapa_texto["FUENTE_DIG"].unique()))
bitacora.registrar("limpieza", "Se normaliza el texto (codificación, mayúsculas y espacios)",
                   f"{sum(cambios_texto.values())} valores corregidos ("
                   + ", ".join(f"{c}: {n}" for c, n in cambios_texto.items())
                   + "): 'FOTOGRAMETR├ìA' → 'FOTOGRAMETRÍA', 'LiDAR ANDINA' → 'LIDAR ANDINA' y espacios dobles en nombres")

Valores corregidos por columna: {'NOMBRE': 5, 'FUENTE_DIG': 6, 'NOM_CUEN': 128}
FUENTE_DIG: ['EIA ANDINA 244 IMG:RAPIDEYE', 'FOTOGRAMETRÍA', 'GE01', 'GOOGLE EARTH', 'GPR Y DRON', 'LANDSAT OLI', 'LIDAR', 'LIDAR ANDINA', 'PLANETSCOPE', 'PLEIADES', 'RAPIDEYE', 'SAF GEOTEC 1:50.000', 'SENTINEL 2', 'SPOT 6', 'SPOT-5 GOOGLE EARTH', 'VUELO HYCON', 'WV02', 'WV03', 'WV04']
[F2 · limpieza] Se normaliza el texto (codificación, mayúsculas y espacios) — 139 valores corregidos (NOMBRE: 5, FUENTE_DIG: 6, NOM_CUEN: 128): 'FOTOGRAMETR├ìA' → 'FOTOGRAMETRÍA', 'LiDAR ANDINA' → 'LIDAR ANDINA' y espacios dobles en nombres


### 3.3 Casting de tipos

- **Coordenadas.** `LATITUD` y `LONGITUD` pasan de texto con coma decimal a `float`. Cualquier valor que no se pueda convertir quedaría como `NaN` contable, nunca como un valor inventado.
- **Fecha de la fuente.** Hay tres formatos (§2.4). Se interpreta primero DD/MM/AAAA y luego MM/DD/AAAA; los registros que traen solo el año quedan sin fecha completa (`NaT`). Se deriva `FUEN_ANIO`, el único dato común a todos los formatos. No se inventan día ni mes.
- **Códigos.** Los códigos de cuenca (`COD_*`), los administrativos (`CUT_*`) y los de clasificación WGI identifican categorías, no cantidades: sumarlos o promediarlos no tiene sentido. Se convierten a texto.
- **Numérica discreta.** `INVE_FECHA` (año del inventario, 5 valores) se mantiene como número entero, porque el orden y la distancia entre años tienen significado.

In [10]:
def convertir_decimal_coma(serie):
    """'-18,17' (texto) → -18.17 (float); lo no interpretable queda como NaN."""
    return pd.to_numeric(serie.astype("string").str.strip().str.replace(",", ".", regex=False), errors="coerce")

def parsear_fecha_mixta(serie):
    """Devuelve (fecha, año): DD/MM/AAAA, luego MM/DD/AAAA; 'AAAA' solo aporta el año."""
    texto = serie.astype("string").str.strip()
    dia_mes = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
    mes_dia = pd.to_datetime(texto, format="%m/%d/%Y", errors="coerce")
    fecha = dia_mes.fillna(mes_dia)
    solo_anio = pd.to_numeric(texto.where(texto.str.fullmatch(r"\d{4}").fillna(False)), errors="coerce")
    anio = fecha.dt.year.astype("Int64").fillna(solo_anio.astype("Int64"))
    return fecha, anio

COLUMNAS_CODIGO = ["COD_CUEN", "COD_SCUEN", "COD_SSCUEN", "CUT_REG", "CUT_PROV", "CUT_COM", "WGI_2", "WGI_3"]

def convertir_tipos(df):
    """Coordenadas a float, fecha mixta a fecha + año y códigos a texto; devuelve una copia."""
    df = df.copy()
    df["LATITUD"] = convertir_decimal_coma(df["LATITUD"])
    df["LONGITUD"] = convertir_decimal_coma(df["LONGITUD"])
    df["FUEN_FECHA"], df["FUEN_ANIO"] = parsear_fecha_mixta(df["FUEN_FECHA"])
    df[COLUMNAS_CODIGO] = df[COLUMNAS_CODIGO].astype("string")
    return df

etapa_tipos = convertir_tipos(etapa_texto)

print("Coordenadas no convertidas:", int(etapa_tipos[["LATITUD", "LONGITUD"]].isna().sum().sum()))
print("Rango LATITUD:", etapa_tipos["LATITUD"].min(), "→", etapa_tipos["LATITUD"].max(),
      "| LONGITUD:", etapa_tipos["LONGITUD"].min(), "→", etapa_tipos["LONGITUD"].max())
print("Fechas completas:", int(etapa_tipos["FUEN_FECHA"].notna().sum()), "| años:", int(etapa_tipos["FUEN_ANIO"].notna().sum()),
      f"({etapa_tipos['FUEN_ANIO'].min()}–{etapa_tipos['FUEN_ANIO'].max()})")
bitacora.registrar("limpieza", "Casting: LATITUD/LONGITUD a float, FUEN_FECHA a fecha + FUEN_ANIO, códigos a texto",
                   f"{int(etapa_tipos['LATITUD'].notna().sum()):,} coordenadas convertidas sin pérdida; "
                   f"{int(etapa_tipos['FUEN_FECHA'].notna().sum()):,} fechas completas y {int(etapa_tipos['FUEN_ANIO'].notna().sum()):,} años "
                   f"({etapa_tipos['FUEN_ANIO'].min()}–{etapa_tipos['FUEN_ANIO'].max()}); {len(COLUMNAS_CODIGO)} columnas de código a texto")

Coordenadas no convertidas: 0
Rango LATITUD: -55.4660836127407 → -17.6620831915806 | LONGITUD: -75.2160394695785 → -67.5430870210823
Fechas completas: 26168 | años: 26180 (1955–2022)
[F2 · limpieza] Casting: LATITUD/LONGITUD a float, FUEN_FECHA a fecha + FUEN_ANIO, códigos a texto — 26,180 coordenadas convertidas sin pérdida; 26,168 fechas completas y 26,180 años (1955–2022); 8 columnas de código a texto


### 3.4 Reconstrucción del volumen

La tabla de §2.3 muestra que el inventario calcula el volumen como `AREA · ESP_MED / 1000 · f_hielo` y el equivalente en agua como `VOL · f_agua`. En los glaciares rocosos, `f_hielo = 0,6` (60 % de hielo) y `f_agua = 0,917`; en los demás tipos, `f_hielo = 1` y `f_agua = 0,85`. Los ceros aparecen solo cuando el resultado es menor que la precisión con que se registró el valor (5 decimales).

**Decisión.** Se recalculan los ceros con la fórmula del propio inventario y se marcan con una bandera. No se imputan con un valor central: la fórmula usa información observada del mismo glaciar. Además, sin este paso la escala logarítmica de §4 daría −∞. Los factores se declaran explícitamente y se verifica con `assert` que coinciden con las medianas medidas.

In [11]:
FACTOR_HIELO = {"GLACIAR ROCOSO": 0.60}    # fracción de hielo; resto de tipos = 1
FACTOR_AGUA = {"GLACIAR ROCOSO": 0.917}    # equivalente en agua; resto de tipos = 0,85
for clase, fila in tabla_volumen.iterrows():
    assert abs(fila["razon_volumen"] - FACTOR_HIELO.get(clase, 1.0)) < 0.01, clase
    assert abs(fila["razon_agua"] - FACTOR_AGUA.get(clase, 0.85)) < 0.01, clase

def reconstruir_volumen(df):
    """Recalcula VOL_km3 = 0 y EQ_AGUAKM3 = 0 con la fórmula del inventario y deja banderas."""
    df = df.copy()
    f_hielo = df["CLASIFICA"].map(FACTOR_HIELO).fillna(1.0)
    f_agua = df["CLASIFICA"].map(FACTOR_AGUA).fillna(0.85)
    vol0 = df["VOL_km3"] == 0
    df.loc[vol0, "VOL_km3"] = (df["AREA_KM2"] * df["ESP_MED"] / 1000 * f_hielo)[vol0]
    eq0 = df["EQ_AGUAKM3"] == 0
    df.loc[eq0, "EQ_AGUAKM3"] = (df["VOL_km3"] * f_agua)[eq0]
    df["VOL_RECALCULADO"] = vol0.astype(int)
    df["EQ_RECALCULADO"] = eq0.astype(int)
    return df

vol_total_antes = etapa_tipos["VOL_km3"].sum()
etapa_volumen = reconstruir_volumen(etapa_tipos)
print("VOL recalculados:", int(etapa_volumen["VOL_RECALCULADO"].sum()), "| valor máximo recalculado:",
      f"{etapa_volumen.loc[etapa_volumen['VOL_RECALCULADO'] == 1, 'VOL_km3'].max():.2e} km³")
print("EQ recalculados:", int(etapa_volumen["EQ_RECALCULADO"].sum()))
print(f"Volumen total: {vol_total_antes:,.5f} → {etapa_volumen['VOL_km3'].sum():,.5f} km³ "
      f"(+{etapa_volumen['VOL_km3'].sum() - vol_total_antes:.6f})")
bitacora.registrar("limpieza", "Se recalculan los ceros de VOL_km3 y EQ_AGUAKM3 con la fórmula del inventario",
                   f"{int(etapa_volumen['VOL_RECALCULADO'].sum())} volúmenes y {int(etapa_volumen['EQ_RECALCULADO'].sum())} equivalentes; "
                   f"volumen total +{etapa_volumen['VOL_km3'].sum() - vol_total_antes:.6f} km³ (despreciable); banderas VOL/EQ_RECALCULADO")

VOL recalculados: 362 | valor máximo recalculado: 5.86e-06 km³
EQ recalculados: 668
Volumen total: 3,026.19653 → 3,026.19786 km³ (+0.001333)
[F2 · limpieza] Se recalculan los ceros de VOL_km3 y EQ_AGUAKM3 con la fórmula del inventario — 362 volúmenes y 668 equivalentes; volumen total +0.001333 km³ (despreciable); banderas VOL/EQ_RECALCULADO


### 3.5 Altitudes: marcar faltantes encubiertos y comparar métodos de imputación

Se marcan como faltantes los **33 `HMIN = 0` de glaciares que terminan en tierra o lago** (se conservan los 36 en fiordo) y el **`HMEDIA` incoherente** del glaciar de Ñuble. Son el 0,13 % de los registros, y eliminarlos podría parecer inocuo; pero eliminar una fila descarta el glaciar completo, con toda su superficie de hielo. Por eso se comparan las alternativas con tres criterios:

1. el **cambio porcentual en la desviación estándar**, porque toda imputación reduce la variabilidad;
2. la **coherencia física**: cuántos valores imputados quedan por encima de `HMEDIA`, algo imposible;
3. la **superficie de hielo que se perdería** al eliminar filas.

El método **por diferencia** aprovecha una variable observada del mismo glaciar: `HMIN = HMEDIA − desnivel mediano (HMEDIA − HMIN)` de su comuna.

In [12]:
def marcar_faltantes_encubiertos(df):
    """HMIN = 0 sin frente en fiordo y HMEDIA < HMIN pasan a NaN para tratarlos explícitamente."""
    df = df.copy()
    hmin0 = (df["HMIN"] == 0) & (df["FRENTE_TER"] != "FIORDO")
    df.loc[hmin0, "HMIN"] = np.nan
    incoherente = df["HMEDIA"] < df["HMIN"]
    df.loc[incoherente, "HMEDIA"] = np.nan
    return df

METODOS_IMPUTACION = ("media", "mediana", "mediana_grupo", "diferencia")

def imputar(df, columna, metodo, grupo=None, referencia=None, minimo=0.0):
    """Imputa los nulos de `columna` y crea la bandera <columna>_IMPUTADO. No modifica `df`."""
    if metodo not in METODOS_IMPUTACION:
        raise ValueError(f"Método desconocido: {metodo}. Opciones: {METODOS_IMPUTACION}")
    if metodo in ("mediana_grupo", "diferencia") and grupo is None:
        raise ValueError(f"El método '{metodo}' requiere `grupo`")
    if metodo == "diferencia" and referencia is None:
        raise ValueError("El método 'diferencia' requiere `referencia`")
    df = df.copy()
    nulos = df[columna].isna()
    if metodo == "media":
        estimado = pd.Series(df[columna].mean(), index=df.index)
    elif metodo == "mediana":
        estimado = pd.Series(df[columna].median(), index=df.index)
    elif metodo == "mediana_grupo":
        estimado = df.groupby(grupo)[columna].transform("median").fillna(df[columna].median())
    else:
        desnivel = df[referencia] - df[columna]
        desnivel_grupo = desnivel.groupby(df[grupo]).transform("median").fillna(desnivel.median())
        estimado = (df[referencia] - desnivel_grupo).clip(lower=minimo)
    df.loc[nulos, columna] = estimado[nulos]
    df[f"{columna}_IMPUTADO"] = nulos.astype(int)
    return df

def comparar_imputaciones(df, columna, opciones, limite_superior):
    """Compara eliminar filas y cada método: dispersión, violaciones físicas y área perdida."""
    desv_observada = df[columna].std()
    nulos = df[columna].isna()
    sin_nulos = df[~nulos]
    filas = [{"metodo": "eliminar filas", "filas": len(sin_nulos), "desv_est": sin_nulos[columna].std(),
              "violaciones": 0, "area_perdida_km2": df.loc[nulos, "AREA_KM2"].sum()}]
    for nombre, parametros in opciones.items():
        r = imputar(df, columna, **parametros)
        filas.append({"metodo": nombre, "filas": len(r), "desv_est": r[columna].std(),
                      "violaciones": int((r.loc[nulos, columna] > r.loc[nulos, limite_superior]).sum()),
                      "area_perdida_km2": 0.0})
    tabla = pd.DataFrame(filas)
    tabla["cambio_desv_pct"] = 100 * (tabla["desv_est"] / desv_observada - 1)
    return tabla.round({"desv_est": 2, "cambio_desv_pct": 4, "area_perdida_km2": 2})

def elegir_metodo(tabla):
    """Regla declarada: sin pérdida de filas, sin violaciones físicas y menor |cambio de desviación|."""
    candidatos = tabla[(tabla["area_perdida_km2"] == 0) & (tabla["violaciones"] == 0)]
    return candidatos.loc[candidatos["cambio_desv_pct"].abs().idxmin(), "metodo"]

etapa_marcada = marcar_faltantes_encubiertos(etapa_volumen)
n_hmin, n_hmedia = int(etapa_marcada["HMIN"].isna().sum()), int(etapa_marcada["HMEDIA"].isna().sum())
print(f"Faltantes marcados: HMIN = {n_hmin} ({100 * n_hmin / len(etapa_marcada):.2f} %), HMEDIA = {n_hmedia}")

OPCIONES_HMIN = {
    "media global": {"metodo": "media"},
    "mediana global": {"metodo": "mediana"},
    "mediana por comuna": {"metodo": "mediana_grupo", "grupo": "COMUNA"},
    "HMEDIA − desnivel mediano por comuna": {"metodo": "diferencia", "grupo": "COMUNA", "referencia": "HMEDIA"},
}
comparacion_hmin = comparar_imputaciones(etapa_marcada.dropna(subset=["HMEDIA"]), "HMIN", OPCIONES_HMIN, "HMEDIA")
display(comparacion_hmin)
metodo_hmin = elegir_metodo(comparacion_hmin)
print("Método elegido según la regla declarada:", metodo_hmin)

Faltantes marcados: HMIN = 33 (0.13 %), HMEDIA = 1


,metodo,filas,desv_est,violaciones,area_perdida_km2,cambio_desv_pct
0,eliminar filas,26146,1260.28,0,967.51,-0.0000
1,media global,26179,1259.48,33,0.00,-0.0630
2,mediana global,26179,1259.56,31,0.00,-0.0568
3,mediana por comuna,26179,1260.08,14,0.00,-0.0155
4,HMEDIA − desnivel mediano por comuna,26179,1260.24,0,0.00,-0.0029


Método elegido según la regla declarada: HMEDIA − desnivel mediano por comuna


**Lectura de la comparación.** Eliminar filas no altera la dispersión, pero descarta glaciares completos y la superficie de hielo que registran. La media y la mediana globales asignan a glaciares patagónicos de baja altitud un valor típico de los Andes centrales, y el resultado queda **por encima de su propia altitud media**, lo que es físicamente imposible. La mediana por comuna reduce el problema, pero no lo elimina. El método por diferencia usa la `HMEDIA` observada de cada glaciar: no produce violaciones y es el que menos modifica la desviación estándar. Es el método elegido según la regla, y la bandera `HMIN_IMPUTADO` registra qué filas se imputaron.

**`HMEDIA` del glaciar de Ñuble.** Con un solo valor, la comparación de dispersión no informa nada. La restricción física define el método: se usa el **punto medio entre `HMIN` y `HMAX`**, que por construcción respeta `HMIN ≤ HMEDIA ≤ HMAX`. La mediana global, en cambio, quedaría bajo su `HMIN`.

In [13]:
def imputar_punto_medio(df, columna, inferior, superior):
    """Imputa los nulos de `columna` con (inferior + superior) / 2 y crea la bandera."""
    df = df.copy()
    nulos = df[columna].isna()
    df.loc[nulos, columna] = ((df[inferior] + df[superior]) / 2)[nulos]
    df[f"{columna}_IMPUTADO"] = nulos.astype(int)
    return df

fila_hmin = comparacion_hmin.set_index("metodo")
mediana_hmedia = etapa_marcada["HMEDIA"].median()
fila_nuble = etapa_marcada["HMEDIA"].isna()
etapa_imputada = imputar_punto_medio(imputar(etapa_marcada, "HMIN", **OPCIONES_HMIN[metodo_hmin]), "HMEDIA", "HMIN", "HMAX")

display(etapa_imputada.loc[fila_nuble, ["COD_GLA", "HMIN", "HMEDIA", "HMAX"]])
print(f"Mediana global de HMEDIA = {mediana_hmedia:,.0f} m < HMIN de ese glaciar → violaría la restricción física")
bitacora.registrar("imputación", f"HMIN = 0 en tierra/lago se marca como faltante e imputa con '{metodo_hmin}'",
                   f"{n_hmin} valores ({100 * n_hmin / len(etapa_imputada):.2f} %); cambio de desv. estándar "
                   f"{fila_hmin.loc[metodo_hmin, 'cambio_desv_pct']:+.4f} %; violaciones de HMIN ≤ HMEDIA: "
                   + ", ".join(f"{m} {int(v)}" for m, v in fila_hmin["violaciones"].items() if m != "eliminar filas")
                   + f"; eliminar filas perdería {fila_hmin.loc['eliminar filas', 'area_perdida_km2']:,.1f} km² de hielo")
bitacora.registrar("imputación", "HMEDIA incoherente se reemplaza por (HMIN + HMAX) / 2",
                   f"{n_hmedia} glaciar; la mediana global ({mediana_hmedia:,.0f} m) violaría HMIN ≤ HMEDIA; "
                   f"el punto medio da {etapa_imputada.loc[fila_nuble, 'HMEDIA'].iloc[0]:,.0f} m")

,COD_GLA,HMIN,HMEDIA,HMAX
26167,CL108116007@,2849.0,2893.0,2937.0


Mediana global de HMEDIA = 1,525 m < HMIN de ese glaciar → violaría la restricción física
[F2 · imputación] HMIN = 0 en tierra/lago se marca como faltante e imputa con 'HMEDIA − desnivel mediano por comuna' — 33 valores (0.13 %); cambio de desv. estándar -0.0029 %; violaciones de HMIN ≤ HMEDIA: media global 33, mediana global 31, mediana por comuna 14, HMEDIA − desnivel mediano por comuna 0; eliminar filas perdería 967.5 km² de hielo
[F2 · imputación] HMEDIA incoherente se reemplaza por (HMIN + HMAX) / 2 — 1 glaciar; la mediana global (1,525 m) violaría HMIN ≤ HMEDIA; el punto medio da 2,893 m


## 4. Transformación por tipo de variable

Cada rol analítico recibe el tratamiento que corresponde a su tipo:

| Rol | Variables | Tratamiento | Por qué |
|---|---|---|---|
| Derivadas | `RANGO_ALT`, `LOG_AREA`, `LOG_VOL`, `CLASE_TAMANO`, `FRENTE_AGUA` | Desnivel, escala log10, clases de tamaño y bandera de frente en agua | La asimetría medida en §2.2 impide comparar en escala lineal; las clases y banderas responden a umbrales del dominio |
| Ordinal | `MZON_GLAC`, `CLASE_TAMANO` | Códigos con **orden declarado por el equipo** | El orden alfabético pondría AUSTRAL < CENTRO < NORTE < SUR, sin sentido geográfico |
| Cíclica | `ORIENTA` | Seno y coseno del ángulo | N (0°) y NW (315°) son vecinas; un código 0–7 las pondría en extremos opuestos |
| Nominal | `CLASIFICA`, `FRENTE_TER`, `REGION` | One-hot | Sin orden natural |
| Alta cardinalidad | `FUENTE_DIG` | Las fuentes con menos de 1 % de los registros se agrupan en `OTRAS` antes del one-hot | Evita columnas casi vacías |
| Identificador | `COD_GLA` | Se conserva como clave y se excluye del análisis | No es una variable |

In [14]:
CORTES_TAMANO = [0, 0.01, 0.1, 1, 10, np.inf]                       # km², escala del dominio
ETIQUETAS_TAMANO = ["< 0,01", "0,01–0,1", "0,1–1", "1–10", "> 10"]
ORDEN_MACROZONA = ["MACROZONA NORTE", "MACROZONA CENTRO", "MACROZONA SUR", "MACROZONA AUSTRAL"]
ANGULOS = {"N": 0, "NE": 45, "E": 90, "SE": 135, "S": 180, "SW": 225, "W": 270, "NW": 315}

def agrupar_categorias_raras(serie, umbral=0.01, etiqueta="OTRAS"):
    """Reemplaza por `etiqueta` las categorías con frecuencia relativa menor que `umbral`."""
    frecuencia = serie.value_counts(normalize=True)
    raras = frecuencia[frecuencia < umbral].index
    return serie.where(~serie.isin(raras), etiqueta)

def derivar_variables(df):
    """Agrega desnivel, logaritmos, clase de tamaño, frente en agua y fuente agrupada."""
    if (df[["AREA_KM2", "VOL_km3"]] <= 0).any().any():
        raise ValueError("El logaritmo requiere AREA_KM2 y VOL_km3 positivos")
    df = df.copy()
    df["RANGO_ALT"] = df["HMAX"] - df["HMIN"]
    df["LOG_AREA"] = np.log10(df["AREA_KM2"])
    df["LOG_VOL"] = np.log10(df["VOL_km3"])
    df["CLASE_TAMANO"] = pd.cut(df["AREA_KM2"], bins=CORTES_TAMANO, labels=ETIQUETAS_TAMANO, right=False)
    df["FRENTE_AGUA"] = df["FRENTE_TER"].isin(["LAGO", "FIORDO"]).astype(int)
    df["FUENTE_GRUPO"] = agrupar_categorias_raras(df["FUENTE_DIG"])
    return df

def codificar_ordinal(serie, orden):
    """Códigos 0..n-1 según `orden`; falla si aparece una categoría no declarada."""
    desconocidas = set(serie.dropna().astype(str)) - set(orden)
    if desconocidas:
        raise ValueError(f"Categorías sin orden declarado: {sorted(desconocidas)}")
    return serie.astype(str).map({c: i for i, c in enumerate(orden)}).astype("Int64")

def codificar_orientacion(serie, plano="PLANO"):
    """Seno y coseno del ángulo de orientación; las laderas planas quedan en (0, 0)."""
    desconocidas = set(serie.dropna()) - set(ANGULOS) - {plano}
    if desconocidas:
        raise ValueError(f"Orientaciones desconocidas: {sorted(desconocidas)}")
    radianes = np.deg2rad(serie.map(ANGULOS).astype(float))
    return pd.DataFrame({"ORIENTA_SIN": np.sin(radianes).fillna(0).round(10),
                         "ORIENTA_COS": np.cos(radianes).fillna(0).round(10)}, index=serie.index)

def codificar_one_hot(df, prefijos):
    """One-hot con una columna binaria (0/1) por categoría; `prefijos` = {columna: prefijo}."""
    return pd.get_dummies(df[list(prefijos)].astype(str), prefix=list(prefijos.values()), prefix_sep="_", dtype=int)

asimetria_antes = etapa_imputada[["AREA_KM2", "VOL_km3"]].skew()
transformado = derivar_variables(etapa_imputada)
transformado["MZON_GLAC"] = pd.Categorical(transformado["MZON_GLAC"], categories=ORDEN_MACROZONA, ordered=True)
asimetria_despues = transformado[["LOG_AREA", "LOG_VOL"]].skew()

ordinales = pd.DataFrame({"MZON_ORD": codificar_ordinal(transformado["MZON_GLAC"], ORDEN_MACROZONA),
                          "TAMANO_ORD": codificar_ordinal(transformado["CLASE_TAMANO"], ETIQUETAS_TAMANO)})
ciclicas = codificar_orientacion(transformado["ORIENTA"])
PREFIJOS_ONE_HOT = {"CLASIFICA": "OH_TIPO", "FRENTE_TER": "OH_FRENTE", "REGION": "OH_REG", "FUENTE_GRUPO": "OH_FUENTE"}
one_hot = codificar_one_hot(transformado, PREFIJOS_ONE_HOT)

print("Asimetría:", {k: round(v, 2) for k, v in asimetria_antes.items()}, "→", {k: round(v, 2) for k, v in asimetria_despues.items()})
display(transformado["CLASE_TAMANO"].value_counts(sort=False).rename("glaciares").to_frame())
print("Orden alfabético:", sorted(ORDEN_MACROZONA), "\nOrden declarado:  ", ORDEN_MACROZONA)
display(transformado[["ORIENTA"]].join(ciclicas).drop_duplicates().sort_values("ORIENTA").reset_index(drop=True))
print("Fuentes agrupadas:", transformado["FUENTE_GRUPO"].value_counts().to_dict())
print("Columnas one-hot:", {p: sum(c.startswith(p + "_") for c in one_hot) for p in PREFIJOS_ONE_HOT.values()})

bitacora.registrar("transformación", "Se derivan RANGO_ALT, LOG_AREA, LOG_VOL, CLASE_TAMANO, FRENTE_AGUA y FUENTE_GRUPO",
                   f"log10 reduce la asimetría del área de {asimetria_antes['AREA_KM2']:.1f} a {asimetria_despues['LOG_AREA']:.2f} "
                   f"y del volumen de {asimetria_antes['VOL_km3']:.1f} a {asimetria_despues['LOG_VOL']:.2f}; cortes de tamaño {CORTES_TAMANO[1:-1]} km²")
bitacora.registrar("transformación", "MZON_GLAC y CLASE_TAMANO se codifican como ordinales con orden declarado",
                   "orden geográfico NORTE → AUSTRAL; el alfabético pondría AUSTRAL < CENTRO < NORTE < SUR")
bitacora.registrar("transformación", "ORIENTA se codifica con seno y coseno",
                   f"variable circular (N = 0°, NW = 315° son vecinas); {int((transformado['ORIENTA'] == 'PLANO').sum())} laderas PLANO → (0, 0)")
bitacora.registrar("transformación", "One-hot para CLASIFICA, FRENTE_TER, REGION y FUENTE_GRUPO",
                   f"nominales sin orden; {one_hot.shape[1]} columnas binarias; FUENTE_DIG: "
                   f"{transformado['FUENTE_DIG'].nunique()} → {transformado['FUENTE_GRUPO'].nunique()} categorías (< 1 % → OTRAS)")

Asimetría: {'AREA_KM2': 43.09, 'VOL_km3': 54.05} → {'LOG_AREA': 0.61, 'LOG_VOL': 0.69}


,glaciares
CLASE_TAMANO,
"< 0,01",5681
"0,01–0,1",12243
"0,1–1",6597
1–10,1446
> 10,213


Orden alfabético: ['MACROZONA AUSTRAL', 'MACROZONA CENTRO', 'MACROZONA NORTE', 'MACROZONA SUR'] 
Orden declarado:   ['MACROZONA NORTE', 'MACROZONA CENTRO', 'MACROZONA SUR', 'MACROZONA AUSTRAL']


,ORIENTA,ORIENTA_SIN,ORIENTA_COS
0,E,1.000000,0.000000
1,N,0.000000,1.000000
2,NE,0.707107,0.707107
3,NW,-0.707107,0.707107
4,PLANO,0.000000,0.000000
5,S,0.000000,-1.000000
6,SE,0.707107,-0.707107
7,SW,-0.707107,-0.707107
8,W,-1.000000,-0.000000


Fuentes agrupadas: {'LANDSAT OLI': 15219, 'SENTINEL 2': 6352, 'WV02': 1814, 'GE01': 1525, 'WV03': 397, 'PLANETSCOPE': 385, 'SPOT 6': 328, 'OTRAS': 160}
Columnas one-hot: {'OH_TIPO': 5, 'OH_FRENTE': 3, 'OH_REG': 16, 'OH_FUENTE': 8}
[F2 · transformación] Se derivan RANGO_ALT, LOG_AREA, LOG_VOL, CLASE_TAMANO, FRENTE_AGUA y FUENTE_GRUPO — log10 reduce la asimetría del área de 43.1 a 0.61 y del volumen de 54.0 a 0.69; cortes de tamaño [0.01, 0.1, 1, 10] km²
[F2 · transformación] MZON_GLAC y CLASE_TAMANO se codifican como ordinales con orden declarado — orden geográfico NORTE → AUSTRAL; el alfabético pondría AUSTRAL < CENTRO < NORTE < SUR
[F2 · transformación] ORIENTA se codifica con seno y coseno — variable circular (N = 0°, NW = 315° son vecinas); 3 laderas PLANO → (0, 0)
[F2 · transformación] One-hot para CLASIFICA, FRENTE_TER, REGION y FUENTE_GRUPO — nominales sin orden; 32 columnas binarias; FUENTE_DIG: 19 → 8 categorías (< 1 % → OTRAS)


## 5. Escalamiento: comparación de los tres escaladores

Las variables continuas se miden en escalas muy distintas: metros de altitud, grados de pendiente, logaritmos de km². Para modelos sensibles a la escala, que podrían usarse en F3, se escalan. Antes de elegir el escalador se mide qué garantiza cada uno y cómo responde a los valores extremos. El primer cuadro aplica los tres al **área sin logaritmo**; el segundo, a las seis variables continuas del modelo.

In [15]:
ESCALADORES = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}
CONTINUAS = ["LOG_AREA", "LOG_VOL", "HMEDIA", "RANGO_ALT", "PENDIENTE", "ESP_MED"]

def escalar(df, columnas, metodo):
    """Escala `columnas` con el método indicado; devuelve (DataFrame *_ESC, escalador ajustado)."""
    if metodo not in ESCALADORES:
        raise ValueError(f"Escalador desconocido: {metodo}. Opciones: {list(ESCALADORES)}")
    if df[columnas].isna().any().any():
        raise ValueError("No se puede escalar con valores nulos: imputar antes")
    escalador = ESCALADORES[metodo]().fit(df[columnas])
    return pd.DataFrame(escalador.transform(df[columnas]), columns=[f"{c}_ESC" for c in columnas], index=df.index), escalador

def comparar_escaladores(df, columnas):
    """Media, desviación, mediana, RIC y % de datos en el 1 % inferior del rango tras cada escalador."""
    filas = []
    for metodo in ESCALADORES:
        escalado, _ = escalar(df, columnas, metodo)
        for columna in escalado:
            x = escalado[columna]
            q1, q3 = x.quantile([0.25, 0.75])
            filas.append({"escalador": metodo, "variable": columna, "media": x.mean(), "desv": x.std(),
                          "mediana": x.median(), "ric": q3 - q1,
                          "pct_en_1pct_inferior": 100 * (x <= x.min() + 0.01 * (x.max() - x.min())).mean()})
    return pd.DataFrame(filas).round(3)

display(comparar_escaladores(transformado, ["AREA_KM2"]))
comparacion_escala = comparar_escaladores(transformado, CONTINUAS)
display(comparacion_escala.pivot(index="variable", columns="escalador", values=["mediana", "ric"]))

,escalador,variable,media,desv,mediana,ric,pct_en_1pct_inferior
0,standard,AREA_KM2_ESC,0.000,1.000,-0.059,0.011,99.03
1,minmax,AREA_KM2_ESC,0.001,0.017,0.000,0.000,99.03
2,robust,AREA_KM2_ESC,5.437,92.227,0.000,1.000,99.03


mediana                    ric                
escalador      minmax robust standard minmax robust standard
variable                                                    
ESP_MED_ESC     0.022    0.0   -0.276  0.029    1.0    0.670
HMEDIA_ESC      0.223    0.0   -0.329  0.115    1.0    0.620
LOG_AREA_ESC    0.264    0.0   -0.120  0.194    1.0    1.377
LOG_VOL_ESC     0.263    0.0   -0.139  0.189    1.0    1.363
PENDIENTE_ESC   0.333    0.0   -0.001  0.134    1.0    1.270
RANGO_ALT_ESC   0.031    0.0   -0.312  0.049    1.0    0.741

**Lectura.** Sobre el área sin transformar, cualquier escalador lineal deja casi todos los glaciares comprimidos en el 1 % inferior del rango. **El problema es la asimetría, no el escalador.** Por eso primero se aplica log10 (§4) y luego se escala.

Entre los tres escaladores:
- **StandardScaler** centra con la media, que los valores extremos desplazan.
- **MinMaxScaler** depende del mínimo y del máximo, es decir, de dos valores extremos.
- **RobustScaler** centra con la mediana y escala con el rango intercuartílico.

**Se elige RobustScaler**, por coherencia con el resto del pipeline. La misma evidencia que llevó a conservar los atípicos legítimos (§2.2) y a imputar con medianas (§3.5) lleva a descartar la media para centrar.

In [16]:
comparacion_area = comparar_escaladores(transformado, ["AREA_KM2"]).set_index("escalador")
pct_minmax_area = comparacion_area.loc["minmax", "pct_en_1pct_inferior"]
escaladas, escalador_robusto = escalar(transformado, CONTINUAS, "robust")
display(escaladas.describe().loc[["50%", "25%", "75%"]].round(4))
bitacora.registrar("escalamiento", "Se usa RobustScaler (mediana 0, RIC 1) sobre 6 variables continuas",
                   f"{', '.join(CONTINUAS)}; coherente con la imputación por mediana y con los atípicos legítimos; "
                   f"sobre el área sin log, MinMax deja el {pct_minmax_area:.1f} % de los datos en el 1 % inferior del rango")

,LOG_AREA_ESC,LOG_VOL_ESC,HMEDIA_ESC,RANGO_ALT_ESC,PENDIENTE_ESC,ESP_MED_ESC
50%,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,-0.4515,-0.4399,-0.5155,-0.3364,-0.5092,-0.3401
75%,0.5485,0.5601,0.4845,0.6636,0.4908,0.6599


[F2 · escalamiento] Se usa RobustScaler (mediana 0, RIC 1) sobre 6 variables continuas — LOG_AREA, LOG_VOL, HMEDIA, RANGO_ALT, PENDIENTE, ESP_MED; coherente con la imputación por mediana y con los atípicos legítimos; sobre el área sin log, MinMax deja el 99.0 % de los datos en el 1 % inferior del rango


## 6. Validación técnica y pruebas

### 6.1 Ensamblaje de los dos conjuntos de salida

- **`glaciares_limpio`**: variables limpias e interpretables, en sus unidades físicas. Es la entrada de los resúmenes descriptivos de F3 y de las visualizaciones de F4.
- **`glaciares_modelo`**: matriz 100 % numérica (ordinales, cíclicas, one-hot, banderas y continuas escaladas), con `COD_GLA` como clave. Queda disponible para un eventual análisis algorítmico en F3.

In [17]:
BANDERAS = ["EN_CAMPO_HIELO", "FRENTE_AGUA", "VOL_RECALCULADO", "EQ_RECALCULADO", "HMIN_IMPUTADO", "HMEDIA_IMPUTADO"]
COLUMNAS_ANALISIS = ["COD_GLA", "REGION", "MZON_GLAC", "CLASIFICA", "AREA_KM2", "VOL_km3", "EQ_AGUAKM3",
                     "HMIN", "HMEDIA", "HMAX", "LATITUD", "LONGITUD", "FRENTE_TER", "ORIENTA", "PENDIENTE", "ESP_MED"]

limpio = transformado.copy()
for columna in ["CLASIFICA", "REGION", "FRENTE_TER", "FUENTE_GRUPO"]:
    limpio[columna] = limpio[columna].astype("category")
modelo = pd.concat([limpio[["COD_GLA"]], ordinales, ciclicas, limpio[BANDERAS], escaladas, one_hot], axis=1)

print("glaciares_limpio:", limpio.shape, "| glaciares_modelo:", modelo.shape)

glaciares_limpio: (26180, 48) | glaciares_modelo: (26180, 49)


### 6.2 Validación del dataset resultante

Cada comprobación verifica una **propiedad específica** del resultado. `ejecutar_validaciones` las ejecuta todas y reporta OK o FALLA sin detenerse en la primera; luego un `assert` exige que todas pasen. Los rangos de coordenadas corresponden al territorio de Chile continental e insular con glaciares: latitudes entre −56° y −17°, longitudes entre −76° y −66°.

In [18]:
def suma_one_hot_es_uno(df, prefijo):
    columnas = [c for c in df.columns if c.startswith(prefijo + "_")]
    return len(columnas) > 0 and bool(df[columnas].sum(axis=1).eq(1).all())

def ejecutar_validaciones(limpio, modelo, original):
    escaladas = modelo[[f"{c}_ESC" for c in CONTINUAS]]
    q1, q3 = escaladas.quantile(0.25), escaladas.quantile(0.75)
    comprobaciones = {
        "Se conservan las 26.180 filas": len(limpio) == len(modelo) == len(original),
        "COD_GLA único, sin nulos y alineado entre conjuntos": limpio["COD_GLA"].is_unique and limpio["COD_GLA"].notna().all()
                                                               and limpio["COD_GLA"].equals(modelo["COD_GLA"]),
        "Sin filas duplicadas": not limpio.duplicated().any(),
        "Sin nulos en columnas de análisis (limpio)": bool(limpio[COLUMNAS_ANALISIS].notna().all().all()),
        "Sin nulos en la matriz del modelo": bool(modelo.notna().all().all()),
        "Matriz del modelo 100 % numérica (salvo la clave)": all(pd.api.types.is_numeric_dtype(modelo[c]) for c in modelo if c != "COD_GLA"),
        **{f"One-hot {p} suma 1 por fila": suma_one_hot_es_uno(modelo, p) for p in PREFIJOS_ONE_HOT.values()},
        "Ordinales dentro del rango declarado": modelo["MZON_ORD"].between(0, 3).all() and modelo["TAMANO_ORD"].between(0, 4).all(),
        "RobustScaler: mediana 0 y RIC 1": bool(np.allclose(escaladas.median(), 0, atol=1e-9) and np.allclose(q3 - q1, 1, atol=1e-9)),
        "Magnitudes físicas positivas (área, volumen, agua)": bool((limpio[["AREA_KM2", "VOL_km3", "EQ_AGUAKM3"]] > 0).all().all()),
        "Coherencia altitudinal HMIN ≤ HMEDIA ≤ HMAX": bool(((limpio["HMIN"] <= limpio["HMEDIA"]) & (limpio["HMEDIA"] <= limpio["HMAX"])).all()),
        "Coordenadas dentro del territorio de Chile": bool(limpio["LATITUD"].between(-56, -17).all() and limpio["LONGITUD"].between(-76, -66).all()),
        "Banderas coinciden con lo registrado": (int(limpio["HMIN_IMPUTADO"].sum()) == n_hmin and int(limpio["HMEDIA_IMPUTADO"].sum()) == n_hmedia
                                                and int(limpio["VOL_RECALCULADO"].sum()) == int((original["VOL_km3"] == 0).sum())),
        "data/raw no fue modificado (SHA-256)": huella_sha256(ENTRADA) == huella_inicial,
    }
    return pd.DataFrame({"validacion": list(comprobaciones), "resultado": ["OK" if v else "FALLA" for v in comprobaciones.values()]})

validaciones = ejecutar_validaciones(limpio, modelo, original)
display(validaciones)
assert (validaciones["resultado"] == "OK").all(), "Hay validaciones fallidas"
bitacora.registrar("validación", f"Se ejecutan {len(validaciones)} validaciones del dataset resultante",
                   f"{(validaciones['resultado'] == 'OK').sum()}/{len(validaciones)} OK: filas, clave, nulos, tipos, one-hot, "
                   "ordinales, RobustScaler, reglas físicas, banderas y huella de data/raw")

,validacion,resultado
0,Se conservan las 26.180 filas,OK
1,"COD_GLA único, sin nulos y alineado entre conj...",OK
2,Sin filas duplicadas,OK
3,Sin nulos en columnas de análisis (limpio),OK
4,Sin nulos en la matriz del modelo,OK
5,Matriz del modelo 100 % numérica (salvo la clave),OK
6,One-hot OH_TIPO suma 1 por fila,OK
7,One-hot OH_FRENTE suma 1 por fila,OK
8,One-hot OH_REG suma 1 por fila,OK
9,One-hot OH_FUENTE suma 1 por fila,OK


[F2 · validación] Se ejecutan 17 validaciones del dataset resultante — 17/17 OK: filas, clave, nulos, tipos, one-hot, ordinales, RobustScaler, reglas físicas, banderas y huella de data/raw


### 6.3 Pruebas de las funciones: casos normales, límite y excepciones

Las funciones del pipeline se prueban sobre un **mini inventario sintético** de cuatro glaciares que reproduce los problemas reales: volumen 0, `HMIN` nulo, coordenadas con coma, fechas en tres formatos y codificación rota. Así cada prueba se ejecuta en milisegundos y su resultado esperado se conoce de antemano. Las excepciones verifican que una entrada inválida produce **el error correcto** en lugar de un resultado silenciosamente erróneo.

In [19]:
def mini_inventario():
    return pd.DataFrame({
        "COD_GLA": ["A", "B", "C", "D"],
        "CLASIFICA": ["GLACIARETE", "GLACIAR ROCOSO", "GLACIARETE", "GLACIAR DE VALLE"],
        "AREA_KM2": [0.001, 0.5, 0.002, 10.0], "ESP_MED": [3.0, 20.0, 4.0, 100.0],
        "VOL_km3": [0.0, 0.006, 0.0, 1.0], "EQ_AGUAKM3": [0.0, 0.0055, 0.0, 0.85],
        "HMIN": [np.nan, 1000.0, 1200.0, 500.0], "HMEDIA": [900.0, 1100.0, 1300.0, 800.0],
        "HMAX": [1000.0, 1200.0, 1400.0, 1500.0], "COMUNA": ["X", "X", "Y", "Y"],
    })

def espera_error(tipo_error, funcion, *args, **kwargs):
    try:
        funcion(*args, **kwargs)
    except tipo_error:
        return
    raise AssertionError(f"Se esperaba {tipo_error.__name__}")

def prueba(condicion, mensaje="resultado inesperado"):
    assert condicion, mensaje

PRUEBAS = [
    ("normal", "convertir_decimal_coma('-33,45') = −33,45",
     lambda: prueba(convertir_decimal_coma(pd.Series(["-33,45"])).iloc[0] == -33.45)),
    ("normal", "reparar_codificacion repara 'FOTOGRAMETR├ìA'",
     lambda: prueba(reparar_codificacion("FOTOGRAMETR├ìA") == "FOTOGRAMETRÍA")),
    ("normal", "parsear_fecha_mixta interpreta DD/MM, MM/DD y año",
     lambda: prueba(parsear_fecha_mixta(pd.Series(["31/12/2019", "03/27/2016", "2019"]))[1].tolist() == [2019, 2016, 2019])),
    ("normal", "reconstruir_volumen aplica AREA·ESP/1000 y marca la bandera",
     lambda: prueba(np.isclose(reconstruir_volumen(mini_inventario()).loc[0, "VOL_km3"], 0.001 * 3 / 1000)
                    and reconstruir_volumen(mini_inventario())["VOL_RECALCULADO"].tolist() == [1, 0, 1, 0])),
    ("normal", "imputar 'diferencia' respeta HMIN ≤ HMEDIA",
     lambda: prueba((lambda r: r.loc[0, "HMIN"] <= r.loc[0, "HMEDIA"] and r["HMIN_IMPUTADO"].sum() == 1)(
         imputar(mini_inventario(), "HMIN", "diferencia", grupo="COMUNA", referencia="HMEDIA")))),
    ("normal", "codificar_ordinal respeta el orden declarado (no alfabético)",
     lambda: prueba(codificar_ordinal(pd.Series(["MACROZONA AUSTRAL", "MACROZONA NORTE"]), ORDEN_MACROZONA).tolist() == [3, 0])),
    ("normal", "La función no modifica el DataFrame de entrada",
     lambda: prueba((lambda m: (imputar(m, "HMIN", "mediana"), m["HMIN"].isna().sum() == 1)[1])(mini_inventario()))),
    ("límite", "imputar sobre una columna sin nulos no cambia nada",
     lambda: prueba((lambda r: r["HMAX"].equals(mini_inventario()["HMAX"]) and r["HMAX_IMPUTADO"].sum() == 0)(
         imputar(mini_inventario(), "HMAX", "mediana")))),
    ("límite", "one-hot con una sola categoría produce una columna de unos",
     lambda: prueba(codificar_one_hot(pd.DataFrame({"X": ["A", "A"]}), {"X": "OH_X"}).to_dict("list") == {"OH_X_A": [1, 1]})),
    ("límite", "RobustScaler sobre una columna constante (RIC = 0) no falla",
     lambda: prueba((escalar(pd.DataFrame({"c": [5.0, 5.0, 5.0]}), ["c"], "robust")[0]["c_ESC"] == 0).all())),
    ("límite", "Orientación PLANO se codifica como (0, 0)",
     lambda: prueba(codificar_orientacion(pd.Series(["PLANO"])).iloc[0].tolist() == [0.0, 0.0])),
    ("límite", "Funciones sobre un conjunto vacío devuelven un resultado vacío",
     lambda: prueba(normalizar_texto(pd.Series([], dtype="str")).empty and codificar_orientacion(pd.Series([], dtype="str")).empty)),
    ("excepción", "imputar con un método desconocido → ValueError",
     lambda: espera_error(ValueError, imputar, mini_inventario(), "HMIN", "moda")),
    ("excepción", "imputar 'mediana_grupo' sin grupo → ValueError",
     lambda: espera_error(ValueError, imputar, mini_inventario(), "HMIN", "mediana_grupo")),
    ("excepción", "escalar con nulos → ValueError",
     lambda: espera_error(ValueError, escalar, mini_inventario(), ["HMIN"], "robust")),
    ("excepción", "escalador desconocido → ValueError",
     lambda: espera_error(ValueError, escalar, mini_inventario(), ["HMAX"], "zscore")),
    ("excepción", "codificar_ordinal con categoría no declarada → ValueError",
     lambda: espera_error(ValueError, codificar_ordinal, pd.Series(["MACROZONA OESTE"]), ORDEN_MACROZONA)),
    ("excepción", "derivar_variables con volumen 0 (log indefinido) → ValueError",
     lambda: espera_error(ValueError, derivar_variables, mini_inventario().assign(VOL_km3=0.0))),
    ("excepción", "eliminar_columnas inexistentes → KeyError",
     lambda: espera_error(KeyError, eliminar_columnas, mini_inventario(), ["NO_EXISTE"])),
    ("excepción", "Bitacora.registrar sin evidencia → ValueError",
     lambda: espera_error(ValueError, Bitacora("prueba").registrar, "etapa", "decisión", "")),
    ("excepción", "localizar_raiz sin data/raw → FileNotFoundError",
     lambda: espera_error(FileNotFoundError, localizar_raiz, Path("/"))),
]

def ejecutar_pruebas(pruebas):
    filas = []
    for tipo, nombre, funcion in pruebas:
        try:
            funcion()
            resultado = "OK"
        except AssertionError as error:
            resultado = f"FALLA: {error}"
        filas.append({"tipo": tipo, "prueba": nombre, "resultado": resultado})
    return pd.DataFrame(filas)

resultado_pruebas = ejecutar_pruebas(PRUEBAS)
display(resultado_pruebas)
assert (resultado_pruebas["resultado"] == "OK").all(), "Hay pruebas fallidas"
conteo = resultado_pruebas["tipo"].value_counts()
bitacora.registrar("validación", f"Se ejecutan {len(resultado_pruebas)} pruebas de las funciones",
                   f"{len(resultado_pruebas)}/{len(resultado_pruebas)} OK: {conteo.get('normal', 0)} casos normales, "
                   f"{conteo.get('límite', 0)} casos límite y {conteo.get('excepción', 0)} excepciones")

,tipo,prueba,resultado
0,normal,"convertir_decimal_coma('-33,45') = −33,45",OK
1,normal,reparar_codificacion repara 'FOTOGRAMETR├ìA',OK
2,normal,"parsear_fecha_mixta interpreta DD/MM, MM/DD y año",OK
3,normal,reconstruir_volumen aplica AREA·ESP/1000 y mar...,OK
4,normal,imputar 'diferencia' respeta HMIN ≤ HMEDIA,OK
5,normal,codificar_ordinal respeta el orden declarado (...,OK
6,normal,La función no modifica el DataFrame de entrada,OK
7,límite,imputar sobre una columna sin nulos no cambia ...,OK
8,límite,one-hot con una sola categoría produce una col...,OK
9,límite,RobustScaler sobre una columna constante (RIC ...,OK


[F2 · validación] Se ejecutan 21 pruebas de las funciones — 21/21 OK: 7 casos normales, 5 casos límite y 9 excepciones


## 7. Persistencia, verificación y bitácora

Los resultados se guardan en `data/processed/`, **nunca sobre `data/raw/`**, y se vuelven a leer para comprobar forma, columnas y tipos: un archivo guardado sin comprobar puede descubrirse corrupto recién en F3. La bitácora se exporta a `docs/bitacora_f2.md`, junto con el perfil de calidad del conjunto limpio.

In [20]:
SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_LIMPIO = SALIDA / "glaciares_limpio.csv"
RUTA_MODELO = SALIDA / "glaciares_modelo.csv"
RUTA_PERFIL = SALIDA / "perfil_glaciares_limpio.csv"
RUTA_BITACORA = DOCS / "bitacora_f2.md"

limpio.to_csv(RUTA_LIMPIO, index=False, date_format="%Y-%m-%d")
modelo.to_csv(RUTA_MODELO, index=False)
perfil_final = perfil_columnas(limpio)
perfil_final.to_csv(RUTA_PERFIL, index=False)

relectura_limpio = pd.read_csv(RUTA_LIMPIO, low_memory=False)
relectura_modelo = pd.read_csv(RUTA_MODELO)
assert relectura_limpio.shape == limpio.shape and list(relectura_limpio.columns) == list(limpio.columns)
assert relectura_modelo.shape == modelo.shape and list(relectura_modelo.columns) == list(modelo.columns)
assert all(pd.api.types.is_numeric_dtype(relectura_modelo[c]) for c in relectura_modelo if c != "COD_GLA")
assert relectura_limpio["COD_GLA"].equals(limpio["COD_GLA"])
assert huella_sha256(ENTRADA) == huella_inicial

for ruta, df in ((RUTA_LIMPIO, limpio), (RUTA_MODELO, modelo)):
    print(f"{ruta.relative_to(RAIZ)}: {df.shape[0]:,} × {df.shape[1]} · {ruta.stat().st_size / 1024**2:.1f} MB")
bitacora.registrar("persistencia", "Se guardan y releen glaciares_limpio.csv y glaciares_modelo.csv en data/processed",
                   f"limpio {limpio.shape[0]}×{limpio.shape[1]} y modelo {modelo.shape[0]}×{modelo.shape[1]}; "
                   "forma, columnas y tipos comprobados tras releer; data/raw intacto")
bitacora.exportar_markdown(RUTA_BITACORA)
print(f"Bitácora: {len(bitacora)} decisiones → {RUTA_BITACORA.relative_to(RAIZ)}")

data/processed/glaciares_limpio.csv: 26,180 × 48 · 10.1 MB
data/processed/glaciares_modelo.csv: 26,180 × 49 · 5.7 MB
[F2 · persistencia] Se guardan y releen glaciares_limpio.csv y glaciares_modelo.csv en data/processed — limpio 26180×48 y modelo 26180×49; forma, columnas y tipos comprobados tras releer; data/raw intacto
Bitácora: 32 decisiones → docs/bitacora_f2.md


### 7.1 Bitácora completa de la Fase 2

In [21]:
with pd.option_context("display.max_colwidth", None):
    display(bitacora.tabla())

,n,etapa,decision,evidencia
0,1,obtención,Se carga el CSV oficial desde data/raw sin modificarlo,"26,180 filas × 45 columnas; SHA-256 5691de5d7d7f… = referencia"
1,2,exploración,Id no aporta información (valor único),0.0 % faltante; 1 valor(es) único(s); valor = 0
2,3,exploración,DATUM no aporta información (valor único),0.0 % faltante; 1 valor(es) único(s); valor = WGS 84
3,4,exploración,HUSO no aporta información (valor único),0.0 % faltante; 1 valor(es) único(s); valor = 19 SUR
4,5,exploración,CH_CASQ no aporta información (faltantes > 60%),97.24 % faltante; 6 valor(es) único(s)
5,6,exploración,CUBIERTO no aporta información (faltantes > 60%),100.0 % faltante; 0 valor(es) único(s)
6,7,exploración,F_ESP_MED no aporta información (valor único),0.0 % faltante; 1 valor(es) único(s); valor = CHEN & OHMURA 1990
7,8,exploración,OBSERVACIO no aporta información (faltantes > 60%),100.0 % faltante; 0 valor(es) único(s)
8,9,exploración,COD_GLA es el identificador; OBJECTID no identifica filas,"COD_GLA único en 26,180 filas; OBJECTID repetido en 102 filas; 0 filas duplicadas"
9,10,exploración,"Área y volumen son muy asimétricos, con atípicos legítimos","AREA_KM2: asimetría 43.1, media 0.803 vs. mediana 0.036 km², 14.37 % fuera del IQR; VOL_km3: asimetría 54.0"


## 8. Trazabilidad y continuidad

Se registran la rama y el commit con que se generaron estas salidas.

**Entrega a la Fase 3.** `data/processed/glaciares_limpio.csv` conserva los 26.180 glaciares con variables limpias en sus unidades físicas, más banderas que indican qué valores fueron imputados o recalculados. Es la entrada de las agregaciones por región, macrozona y clasificación. `glaciares_modelo.csv` queda disponible si F3 requiere una matriz numérica.

**Pendiente de verificación documental:** significado y unidades de `ERROR`, `PERIMETRO`, `PENDIENTE` y los códigos WGI, que la DGA no publica junto al CSV.

**Referencias.** Dirección General de Aguas. (2022). *Inventario público de glaciares de Chile 2022* [Conjunto de datos]. Ministerio de Obras Públicas. · Chen, J., & Ohmura, A. (1990). Estimation of Alpine glacier water resources and their change since the 1870s. *IAHS Publication, 193*, 127–135. · McKinney, W. (2022). *Python for data analysis* (3.ª ed.). O’Reilly Media. · scikit-learn developers. (2026). *Preprocessing data*. https://scikit-learn.org/stable/modules/preprocessing.html

In [22]:
def version_repositorio(raiz):
    """Rama, commit y cambios sin commitear; no falla si Git no está disponible."""
    def git(*args):
        return subprocess.run(["git", *args], cwd=raiz, capture_output=True, text=True, check=True).stdout.strip()
    try:
        return {"rama": git("rev-parse", "--abbrev-ref", "HEAD"), "commit": git("rev-parse", "--short", "HEAD"),
                "cambios_sin_commit": bool(git("status", "--porcelain", "--", ":!data/processed", ":!docs/bitacora_f2.md"))}
    except (OSError, subprocess.CalledProcessError):
        return {"rama": "no disponible", "commit": "no disponible", "cambios_sin_commit": None}

for clave, valor in version_repositorio(RAIZ).items():
    print(f"{clave}: {valor}")
print("F2 finalizada.")

rama: fix-robustez-notebooks
commit: d22431a
cambios_sin_commit: True
F2 finalizada.
